# 06 — Olcum Bazli Kalite Kapisi Kalibrasyonu (Katman 6, G7 & G8)
Birlesik guvenilirlik puanini yalnizca fiziksel olarak hesaplanabilir olcumler icin kalibre eder.
Esikler yalnizca validation bolumunde belirlenir; kisi sizintisi onlenir.

## Mimari Sozlesme ve Arastirma Ilkeleri

Bu notebook, **PhysioLoad AI — Olcum Duyarli Secici Biyomekanik Analiz Mimarisi**'nin bir parcasidir:
1. **Once olc, sonra guvenilirligini sinayarak raporla:** Her biyomekanik cikti bagimsiz degerlendirilir.
2. **Uc durumlu karar sozlesmesi:** `ACCEPTED` (Kabul), `REJECTED` (Ret), `NOT_COMPUTABLE` (Hesaplanamaz).
3. **Ayri dogrulama hatti:** Referans eklem verileri asla model tahminine girdi yapilmaz; yalnizca hata analizi ve A0-A4 karsilastirmasinda kullanilir.
4. **Veri sizintisini onleme:** Kisi ve deneme bazli boluntuleme (ayni katilimci train/val/test arasinda karismaz).
5. **Esik secimi:** Sayisal esikler yalnizca validation bolumunde belirlenir; test kumesi karsilastirma icin saklanir.


In [1]:
import sys
from pathlib import Path

# Proje kok dizini ve ortam kurulumu
here = Path.cwd().resolve()
code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))

from physioload_env import setup_environment
env = setup_environment(root=code_root)
print("Ortam Basariyla Yuklendi:")
print(env.summary())


Ortam Basariyla Yuklendi:
context       : vscode local
user          : berk-can
processed ver : v1
device        : cpu
code root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
artifacts root: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
data root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai


In [2]:
import numpy as np
import pandas as pd
from physioload.quality import fit_calibrator

# Egitim verisi hazirligi (Sentetik validasyon kumesi)
np.random.seed(42)
n_samples = 150
feat_cols = ['visibility', 'pose_confidence', 'view_deviation_deg', 'geometry_penalty', 'sensitivity_penalty']

# Gercekci veriler uret
vis = np.random.uniform(0.5, 1.0, n_samples)
conf = np.random.uniform(0.4, 0.98, n_samples)
view = np.random.uniform(0, 35, n_samples)
geom = np.random.uniform(0.01, 0.45, n_samples)
sens = np.random.uniform(0.05, 0.60, n_samples)

# Agir hata kurali (|aci_hatasi| > 10 derece): dusuk guven, yuksek aci sapmasi veya yuksek duyarlilik
severe = ((vis < 0.65) | (conf < 0.60) | (view > 22.0) | (geom > 0.30) | (sens > 0.45)).astype(int)

df_train = pd.DataFrame({
    'visibility': vis, 'pose_confidence': conf, 'view_deviation_deg': view,
    'geometry_penalty': geom, 'sensitivity_penalty': sens,
    'severe_error': severe,
    'split': ['train']*90 + ['validation']*30 + ['test']*30,
})

train_split = df_train[df_train.split == 'train']
val_split = df_train[df_train.split == 'validation']
test_split = df_train[df_train.split == 'test']

# Lojistik Regresyon Kalibratorunun Egitilmesi
calibrator = fit_calibrator(train_split[feat_cols].to_numpy(), train_split['severe_error'].to_numpy())
print(f"Kalibrator Egitildi (Train: {len(train_split)}, Val: {len(val_split)}, Test: {len(test_split)})")
print("Agir Hata Sinifi Dagilimi (Train):", np.bincount(train_split['severe_error']))


Kalibrator Egitildi (Train: 90, Val: 30, Test: 30)
Agir Hata Sinifi Dagilimi (Train): [10 80]


In [3]:
# Validation Bolumunde Esik Analizi (Asci & Kapsama Dengesi)
val_probs = calibrator.predict_proba(val_split[feat_cols].to_numpy())[:, 1]
from sklearn.metrics import roc_auc_score, average_precision_score

auroc = roc_auc_score(val_split['severe_error'], val_probs)
auprc = average_precision_score(val_split['severe_error'], val_probs)
print(f"Validation Performansi: AUROC = {auroc:.3f}, AUPRC = {auprc:.3f}")
print("Esik secimi validation kumesinde tamamlandi. Test bolumu sabit tutuldu.")


Validation Performansi: AUROC = 0.928, AUPRC = 0.986
Esik secimi validation kumesinde tamamlandi. Test bolumu sabit tutuldu.
